# Impact Analysis Workflow

## Executive summary
Intersect the Copernicus impact layers cached by `01_cems_api_ingestion` with GADM units, OSM infrastructure, and WorldPop. Population exposure is counted on raster cells that touch damaged features.

## Prerequisites
Run `01_cems_api_ingestion` first so `data/cache/cems/{activation_id}/` contains the vector GeoJSON. GADM, Overpass, and WorldPop require network access. `bbox` may stay null; it is taken from those cached layers.

## Parameters

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.cems_api import vector_cache_dir
from src.copernicus_loader import load_params

params = load_params(PROJECT_ROOT)
ACTIVATION_ID = params['activation_id']
COUNTRY_CODE = params['country_code']
ADMIN_LEVEL = int(params['target_admin_level'])
CEMS_CACHE = vector_cache_dir(ACTIVATION_ID)
POPULATION_RASTER = PROJECT_ROOT / 'data' / 'baseline' / 'population.tif'
print({'activation_id': ACTIVATION_ID, 'country_code': COUNTRY_CODE, 'cems_cache': str(CEMS_CACHE)})

{'activation_id': 'EMSR884', 'country_code': 'VEN', 'cems_cache': 'C:\\Users\\loai\\Dropbox\\20251211-Action\\IFRC Project\\usecase2\\notebooks\\montandon_impact_estimation\\data\\cache\\cems\\EMSR884'}


## Load cached Copernicus impact and baseline layers
The cache keeps every monitoring repeat. This step uses the latest layer for each area and theme, skips `notAnalysed`, then applies the damage grades in `params.yaml`.

In [2]:
import importlib

from shapely.geometry import box
from src.baseline_fetcher import fetch_admin_boundaries, fetch_osm_infrastructure, fetch_worldpop_population
from src.cems_api import load_cached_vectors
import src.copernicus_loader as copernicus_loader

importlib.reload(copernicus_loader)
from src.copernicus_loader import bbox_from_geodataframe, concat_layers, filter_impact

vector_layers = load_cached_vectors(ACTIVATION_ID)
impact_all = concat_layers(vector_layers)
impact = filter_impact(impact_all, params)
if impact.empty:
    raise ValueError('No impact features remain after the damage filter')

buffer_deg = float((params.get('impact') or {}).get('bbox_buffer_deg', 0.05))
BBOX = params.get('bbox') or bbox_from_geodataframe(impact, buffer_deg)

admin = fetch_admin_boundaries(
    country_code=COUNTRY_CODE,
    admin_level=ADMIN_LEVEL,
    source=params['baseline']['admin_source'],
)
admin = admin[admin.to_crs('EPSG:4326').intersects(box(*BBOX))].copy()

infrastructure = fetch_osm_infrastructure(
    bbox=BBOX,
    infrastructure_filters=params['baseline']['infrastructure_filters'],
)
if infrastructure.attrs.get('overpass_errors'):
    print('Overpass warnings:', infrastructure.attrs['overpass_errors'])

population_path = None
try:
    population_path = fetch_worldpop_population(
        country_code=COUNTRY_CODE,
        year=int(params['baseline']['worldpop_year']),
        bbox=BBOX,
        output_path=POPULATION_RASTER,
    )
except Exception as exc:
    print(f'WorldPop download failed; continuing without population: {exc}')

print({
    'cached_layers': len(vector_layers),
    'features_loaded': len(impact_all),
    'impact_features': len(impact),
    'bbox': BBOX,
    'admin_units_in_extent': len(admin),
    'infrastructure_features': len(infrastructure),
    'population_raster': str(population_path) if population_path else None,
})

Overpass warnings: ["school: HTTPSConnectionPool(host='overpass.kumi.systems', port=443): Read timed out. (read timeout=120)"]
{'cached_layers': 15, 'features_loaded': 29056, 'impact_features': 3127, 'bbox': [-68.9048408, 10.162394579999999, -66.50240553, 10.677013850000002], 'admin_units_in_extent': 61, 'infrastructure_features': 3820, 'population_raster': 'c:\\Users\\loai\\Dropbox\\20251211-Action\\IFRC Project\\usecase2\\notebooks\\montandon_impact_estimation\\data\\baseline\\population.tif'}


## Exposure by administrative unit

In [3]:
from src.spatial_analysis import (
    admin_id_column,
    admin_name_column,
    choose_analysis_crs,
    clip_features_to_impact,
    population_under_impact,
    summarize_impact_by_admin,
)

analysis_crs = choose_analysis_crs(admin)
id_col = admin_id_column(admin)
name_col = admin_name_column(admin)
impact_admin = summarize_impact_by_admin(impact, admin, target_crs=analysis_crs, id_column=id_col)
exposed_infra = clip_features_to_impact(infrastructure, impact)
if population_path:
    population = population_under_impact(admin, impact, population_path)
    impact_admin = impact_admin.merge(population, on=id_col, how='left')
    impact_admin['pop_exposed'] = impact_admin['pop_exposed'].fillna(0)
    impact_admin['pop_total'] = impact_admin['pop_total'].fillna(0)
else:
    impact_admin['pop_exposed'] = None
    impact_admin['pop_total'] = None

table_cols = [id_col, name_col, 'impact_feature_count', 'impact_area_km2', 'pop_exposed', 'pop_total']
table_cols = [c for c in table_cols if c in impact_admin.columns]
ranked = impact_admin.drop(columns='geometry').sort_values('impact_feature_count', ascending=False)
display(ranked[table_cols].head(20))
print({
    'analysis_crs': analysis_crs,
    'admin_units': len(impact_admin),
    'infrastructure_in_impact': len(exposed_infra),
    'damaged_features': int(impact_admin['impact_feature_count'].sum()),
    'pop_exposed_sum': None if impact_admin['pop_exposed'].isna().all() else float(impact_admin['pop_exposed'].fillna(0).sum()),
})

,GID_2,NAME_2,impact_feature_count,impact_area_km2,pop_exposed,pop_total
47,VEN.23.1_1,Vargas,2802,6.207222,128623.514191,3.328019e+05
50,VEN.24.3_1,Bruzual,101,0.000000,12273.735382,5.110006e+04
16,VEN.7.5_1,JuanJoséMora,96,0.000000,11316.896019,7.285543e+04
58,VEN.24.12_1,Sucre,59,0.000000,3086.409241,2.064018e+04
51,VEN.24.4_1,Cocorote,23,0.000000,6471.248657,4.352058e+04
27,VEN.11.1_1,Libertador,21,0.000000,27009.463867,1.792160e+06
1,VEN.4.3_1,FranciscoLinares,3,0.000000,0.000000,0.000000e+00
7,VEN.4.9_1,MarioBriceñoIragorry,0,0.000000,0.000000,9.436090e+04
8,VEN.4.10_1,OcumaredelaCostadeOro,0,0.000000,0.000000,1.810318e+04
5,VEN.4.7_1,JoséRRevenga,0,0.000000,0.000000,4.756867e+04


{'analysis_crs': 'EPSG:32619', 'admin_units': 61, 'infrastructure_in_impact': 5, 'damaged_features': 3105, 'pop_exposed_sum': 189812.04067230225}


## Map and exports
The map shows admin-level counts, not the raw building layer.

In [4]:
from src.visualization import export_geodataframe, impact_map

point_infra = exposed_infra[exposed_infra.geometry.geom_type.isin(['Point', 'MultiPoint'])]
layers = {
    'Exposed population / damage count': impact_admin,
    'Infrastructure in impact (points)': point_infra,
}
fmap = impact_map(
    layers,
    choropleth_layer='Exposed population / damage count',
    choropleth_column='impact_feature_count',
)
display(fmap)

out_html = PROJECT_ROOT / params['outputs']['map_html']
out_csv = PROJECT_ROOT / params['outputs']['zonal_csv']
out_geojson = PROJECT_ROOT / params['outputs']['zonal_geojson']
out_html.parent.mkdir(parents=True, exist_ok=True)
fmap.save(out_html)
ranked[table_cols].to_csv(out_csv, index=False, encoding='utf-8-sig')
export_geodataframe(impact_admin.to_crs('EPSG:4326'), out_geojson)
print({'map': str(out_html), 'csv': str(out_csv), 'geojson': str(out_geojson)})

{'map': 'c:\\Users\\loai\\Dropbox\\20251211-Action\\IFRC Project\\usecase2\\notebooks\\montandon_impact_estimation\\data\\impact_map.html', 'csv': 'c:\\Users\\loai\\Dropbox\\20251211-Action\\IFRC Project\\usecase2\\notebooks\\montandon_impact_estimation\\data\\impact_by_admin.csv', 'geojson': 'c:\\Users\\loai\\Dropbox\\20251211-Action\\IFRC Project\\usecase2\\notebooks\\montandon_impact_estimation\\data\\impact_by_admin.geojson'}
